<a href="https://colab.research.google.com/github/chadnar2/RAG-MedicalAnalysis/blob/main/Final_Full_Code_NLP_RAG_Project_Notebook_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

Options to solve:
1. query ->llm->response
2. query + system_prompt -> llm ->reponse
3. query + RAG context -> llm -> response
response (gpt 40 mini) -> llm (gpo 4o) -> Evaluate the response

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir --verbose


Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 123.2 MB/s eta 0:00:00
  Running command pip subprocess to install build dependencies
  Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
  Non-user install by explicit request
  Created build tracker: /tmp/pip-build-tracker-4ddvm08q
  Entered build tracker: /tmp/pip-build-tracker-4ddvm08q
  Created temporary directory: /tmp/pip-install-o28wsq2r
  Created temporary directory: /tmp/pip-ephem-wheel-cache-es1kzd0a
  1 location(s) to search for versions of scikit-build-core:
  * https://pypi.org/simple/scikit-build-core/
  Fetching project page and analyzing links: https://pypi.org/simple/scikit-build-core/
  Getting page https://pypi.org/simple/scikit-build-core/
  Found index url https://pypi.org/simple/
  Looking up "https://pypi.org/simple/scikit-build-core/" in the cache
  Request header has "max_age" as 0, cache bypasse

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [ ]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [ ]:
model_path = hf_hub_download(
    repo_id= model_name_or_path,
    filename= model_basename
)

mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

In [ ]:
llm = Llama(
    model_path=model_path,
    n_ctx=4000,
    n_gpu_layers=36,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 1 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [ ]:
def response(query,max_tokens=256,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [ ]:
response("What treatment options are available for managing hypertension?")

'\n\nHypertension, or high blood pressure, is a common condition that can increase the risk of various health problems such as heart disease, stroke, and kidney damage. The good news is that there are several effective treatment options available to help manage hypertension and reduce the risk of complications. Here are some of the most commonly used treatments:\n\n1. Lifestyle modifications: Making lifestyle changes is often the first line of defense against hypertension. This may include eating a healthy diet rich in fruits, vegetables, whole grains, and lean proteins; limiting sodium intake; getting regular physical activity; maintaining a healthy weight; and reducing stress through techniques such as meditation or deep breathing exercises.\n2. Medications: If lifestyle modifications alone are not enough to control hypertension, medications may be necessary. There are several classes of drugs used to treat hypertension, including diuretics, beta blockers, ACE inhibitors, calcium cha

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. ABCs: Ensure airway patency, adequate breathing, and circulatory support. Provide high-flow oxygen via a non-rebreather mask or endotracheal tube if necessary. Initiate intravenous fluids to maintain adequate blood pressure and organ perfusion.\n3. Antibiotics: Administer broad-spectrum antibiotics based on the suspected source of infection and local microbiology data. Consider obtaining cultures before administering antibiotics if feasible.\n4. Source con

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can vary from person to person, but some common signs include:\n\n1. Abdominal pain: The pain is typically located in the lower right quadrant of the abdomen and may start as a mild discomfort that gradually worsens over time. The pain may be constant or intermittent and may be aggravated by movement, deep breathing, or coughing.\n2. Loss of appetite: People with appendicitis may lose their appetite due to abdominal pain or nausea.\n3. Nausea and vomiting: Vomiting is a common symptom of appendicitis, especially in the later stages of the condition.\n4. Fever: A fever of 100.4°F (38°C) or higher may be present in people with appendicitis.\n5. Constipation or diarrhea: Some people with appendicitis experience constipation, while others have diarrhea.\n6. Rebound tenderness: When'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications.\n\nThere are several treatments that have been shown to be effective in addressing sudden patchy hair loss:\n\n1. Corticosteroids: These are anti-inflammatory drugs that can help reduce inflammation and suppress the immune system's attack on the hair follicles. They can be applied topically or taken orally, depending on the severity of the condition.\n2. Minoxidil: This is a medication that has been shown to promote hair growth in some people with alopecia areata. It works by incre

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nA person who has sustained a physical injury to the brain, also known as a traumatic brain injury (TBI), may require various treatments depending on the severity and location of the injury. Here are some common treatments recommended for individuals with TBIs:\n\n1. Emergency care: In case of a severe TBI, the first priority is to provide emergency care to ensure the person's airway is clear, breathing is stable, and circulation is adequate. This may involve intubation, oxygen therapy, and intravenous fluids.\n2. Surgery: Depending on the location and severity of the injury, surgery may be necessary to remove hematomas (clots) or repair skull fractures.\n3. Medications: Various medications may be prescribed to manage symptoms associated with TBIs, such as pain relievers for headaches, anti-seizure medications, and sedatives to help the person relax and rest.\n4. Rehabilitation: Rehabilitation is an essential component of treatment for individuals with TBIs. This may include physic

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sling, or other available materials to immobilize the leg and prevent movement. Be sure not to apply too much pressure on the injury site.\n4. Provide pain relief: Offer over-the-counter pain medication, such as acetaminophen or ibuprofen, to help manage pain.\n5. Seek medical attention: If the fracture is severe or if you suspect that there may be other injuries, call for emergency medical assistance.\n\nOnce you've ensured the person's safety and stabi

## Question Answering using LLM with Prompt Engineering

In [ ]:
system_prompt="""
You are an AI assistant specialized in medical knowlede. Your role is to provide clear, precise, and medically reliable responses based on established medical guidelines and best practices.
When answering, prioritize factual correctness, align with widely accepted medical standards, and ensure clarity for both medical professionals and general users.
If a query requires specic reference materials beyond general knowledge, acknowledge the limitation rather than speculating.
Help clinicians manage chronic diseases and medical precicaments safely and consistently by summarizing and contextualizing the most relevant guideline passages for the specific patient scenario.
Always prioritize patient safety:
Call out red‑flag symptoms and decompensation signs for the conditions and recommend urgent or emergency evaluation when indicated.
"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = system_prompt+"\n"+ "What is the protocol for managing sepsis in a critical care unit?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that can arise from an infection, typically bacterial. In a critical care unit, managing sepsis involves a multidisciplinary approach focusing on early recognition, rapid response, and effective treatment. Here are the key steps:\n\n1. Early Recognition:\n   - Monitor vital signs closely (temperature, heart rate, respiratory rate, blood pressure) every hour.\n   - Assess mental status regularly.\n   - Look for signs of infection (fever, chills, redness, swelling, or pain).\n   - Use a validated scoring system like the Sequential Organ Failure Assessment (SOFA) score to identify patients at risk.\n\n2. Rapid Response:\n   - If sepsis is suspected, initiate broad-spectrum antibiotics as soon as possible.\n   - Begin fluid resuscitation with crystalloids to maintain adequate blood pressure and organ perfusion.\n   - Provide supplemental oxygen if needed to maintain adequate oxygenation.\n   - Initiate vasopressors if necessary to maintain mean a

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = system_prompt+"\n"+ "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the cecum (the beginning of the large intestine). The common symptoms for appendicitis include:\n1. Sudden onset of pain in the lower right abdomen, often starting around the navel and then shifting to the right side.\n2. Loss of appetite and feeling sick to your stomach (nausea) with or without vomiting.\n3. Fever, usually above 101°F (38.3°C).\n4. Abdominal swelling and rigidity in the right lower quadrant.\n5. Pain worsens when walking, coughing, or making other jarring movements.\n\nAppendicitis cannot be cured via medicine alone due to the risk of appendix rupture, which can lead to severe complications such as peritonitis and sepsis. If left untreated, these complications can be life-threatening. Therefore, when diagnosed with appendicitis, surgical intervention is necessary. The standard procedure for treating appendicitis is an appendectomy, which involves rem

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input=system_prompt+"\n"+ "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that results in the sudden loss of hair in small, round patches on the scalp or other areas of the body. The exact cause of alopecia areata is not fully understood, but it's believed to be related to a problem with the immune system.\n\nEffective treatments for addressing sudden patchy hair loss include:\n\n1. Corticosteroids: Topical and injectable corticosteroids are commonly used to treat alopecia areata. They help reduce inflammation and suppress the immune response, allowing hair regrowth.\n2. Immunomodulators: Systemic immunomodulators like minoxidil or methotrexate can be prescribed for more extensive hair loss or when topical treatments are not effective. These medications help slow down the progression of hair loss and promote regrowth.\n3. DHT blockers: Finasteride, a DHT (dihydrotestosterone) blocker, may be used to treat alopecia areata in men. It can help prevent further hair loss and pr

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input=system_prompt+"\n"+ "what treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nI cannot provide an exact answer without more information about the specific type and severity of the brain injury. However, I can suggest some common treatments that may be recommended based on established medical guidelines for various types of brain injuries.\n\n1. Traumatic Brain Injury (TBI): For mild TBIs, rest, hydration, and over-the-counter pain relievers may be sufficient. More severe cases may require hospitalization, surgery, or rehabilitation. Rehabilitation typically includes physical therapy, occupational therapy, speech therapy, and cognitive rehabilitation to help the patient regain lost skills and functions.\n\n2. Stroke: Depending on the type and location of the stroke, treatments may include thrombolytic drugs (such as tPA) to dissolve blood clots, mechanical thrombectomy to remove clots, or medications to manage symptoms and prevent complications (such as anticoagulants, antiplatelet agents, or antihypertensives). Rehabilitation is also crucial for stroke pati

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input=system_prompt+"\n"+ "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nA leg fracture is a serious injury that requires prompt medical attention. Here are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:\n\n1. Assess the severity of the injury: Check for signs of open wounds, deformities, swelling, numbness, or inability to move the limb. If there are any red flags such as severe pain, inability to bear weight, or signs of shock (pale, clammy skin, rapid heartbeat, or shallow breathing), call for emergency medical help immediately.\n2. Immobilize the fracture: Use a splint, sling, or other immobilizing device to prevent further movement and reduce pain. If you don't have access to these items, use whatever is available, such as sticks, branches, or clothing, to create a makeshift splint.\n3. Provide comfort and support: Help the person lie down in a comfortable position with the injured leg elevated to reduce swelling and pain. Offer emotional support and reassurance.\n4. Monitor vital signs

## Data Preparation for RAG

### Loading the Data

Create the context

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
manual_pdf_path = "/content/drive/MyDrive/Project5/medical_diagnosis_manual.pdf"

In [ ]:
pdf_loader = PyMuPDFLoader(manual_pdf_path)

In [ ]:
manual = pdf_loader.load()

In [ ]:
#print(#type(manual), manual)

### Data Overview

#### Checking the first 5 pages

In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

Page Number : 1
chadnar@aim.com
YUF3JRZ527
This file is meant for personal use by chadnar@aim.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
chadnar@aim.com
YUF3JRZ527
This file is meant for personal use by chadnar@aim.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .......................................................................................................................................................

#### Checking the number of pages

In [ ]:
len(manual)

4114

### Data Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1024,
    chunk_overlap= 20
)

In [ ]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
len(document_chunks)

4530

In [ ]:
document_chunks[0].page_content

'chadnar@aim.com\nYUF3JRZ527\nThis file is meant for personal use by chadnar@aim.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [ ]:
document_chunks[2].page_content

"Table of Contents\n1\nFront    ................................................................................................................................................................................................................\n1\nCover    .......................................................................................................................................................................................................\n2\nFront Matter    ...........................................................................................................................................................................................\n53\n1 - Nutritional Disorders    ...............................................................................................................................................................\n53\nChapter 1. Nutrition: General Considerations    ...........................................................................................

In [ ]:
document_chunks[3].page_content

'491\nChapter 44. Foot & Ankle Disorders    .....................................................................................................................................\n502\nChapter 45. Tumors of Bones & Joints    ...............................................................................................................................\n510\n5 - Ear, Nose, Throat & Dental Disorders    ..................................................................................................................\n510\nChapter 46. Approach to the Patient With Ear Problems    ...........................................................................................\n523\nChapter 47. Hearing Loss    .........................................................................................................................................................\n535\nChapter 48. Inner Ear Disorders    ...................................................................................................

In [ ]:
document_chunks[100].page_content

"neurologic deficits to progress.\nEtiology\nInadequate vitamin B12 intake is possible in vegans but is otherwise unlikely. Breastfed babies of vegan\nmothers may develop vitamin B12 deficiency by age 4 to 6 mo because their\n[\nTable 4-5. Causes of Vitamin B12 Deficiency]\nliver stores (which are normally extensive) are limited and their rapid growth rate results in high demand.\nVitamin B12 deficiency usually results from inadequate absorption (see Table 4-5 and p. 153), which, in\nthe elderly, most commonly results from decreased acid secretion. In such cases, crystalline vitamin B12\n(such as that available in vitamin supplements) can be absorbed, but food-bound vitamin B12 is not\nliberated and absorbed normally. Inadequate absorption may occur in blind loop syndrome (with\novergrowth of bacteria) or fish tapeworm infestation; in these cases, bacteria or parasites use ingested\nvitamin B12 so that less is available for absorption. Vitamin B12 absorption may be inadequate if ileal\

In [ ]:
document_chunks[-1].page_content

"Z\nZafirlukast 1879\nZalcitabine 1451\nin children 2854\nZaleplon 1709\nZanamivir 1407\nin influenza 1407, 1929\nZAP-70 (zeta-associated protein 70) deficiency 1092, 1108\nZavanelli maneuver 2680\nZellweger syndrome 2383, 3023\nZenker's diverticulum 125\nZidovudine 1451, 1453\nin children 2854\nZileuton 1881\nin asthma 1880\nZinc 49, 55, 3431-3432\nin common cold 1405\ndeficiency of 11, 49, 55\nin dermatophytoses 705\npoisoning with 3328, 3353\nrecommended dietary allowances for 50\nreference values for 3499\ntoxicity of 49, 55\ncopper deficiency and 49\nin Wilson's disease 52\nZinc oxide 2233\ngelatin formulation of 646, 672\nZinc pyrithione 647\nZinc shakes 55\nZipper injury 3239, 3240\nZiprasidone\nin agitation 1492\nin bipolar disorder 3059\npoisoning with 3347\nin schizophrenia 1566\nZoledronate 359, 361, 848\nZollinger-Ellison syndrome 95, 199, 200-201, 910\nmastocytosis vs 1125\nMenetrier's disease vs 132\npeptic ulcer disease vs 134\nZolmitriptan 1721\nZolpidem 1709, 3103\nZon

### Embedding

In [ ]:
embedding_model = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipykernel_10255/3455502510.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [ ]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  384


True

In [ ]:
embedding_1,embedding_2

([-0.06483092159032822,
  0.05013839527964592,
  -0.0543350875377655,
  -0.02583663910627365,
  0.11583440750837326,
  -0.0458604171872139,
  -0.007056856527924538,
  0.04439462721347809,
  0.0487370491027832,
  0.05635363236069679,
  0.05444062501192093,
  0.10916037112474442,
  0.019592542201280594,
  -0.07427626848220825,
  -0.01725657656788826,
  -0.036708079278469086,
  -0.11354075372219086,
  -0.019937913864850998,
  -0.044995348900556564,
  0.0034850528463721275,
  -0.05934050679206848,
  0.03198868781328201,
  0.05329494550824165,
  0.0023034033365547657,
  0.020423486828804016,
  -0.026730071753263474,
  -0.05363297089934349,
  0.07606585323810577,
  -0.10231149941682816,
  -0.08984718471765518,
  0.008420372381806374,
  0.03537655249238014,
  0.06604979187250137,
  0.021754639223217964,
  0.04487482085824013,
  0.018676135689020157,
  0.006156812887638807,
  -0.03619248420000076,
  -0.037483349442481995,
  -0.0262204147875309,
  -0.03264356032013893,
  -0.029349345713853836,


### Vector Database

In [ ]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [ ]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [ ]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

/tmp/ipykernel_10255/2756559696.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


In [ ]:
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
vectorstore.similarity_search("What are the common symptons and treatments for pulmonary embolism?",k=3)

[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'format': 'PDF 1.7', 'file_path': '/content/drive/MyDrive/Project5/medical_diagnosis_manual.pdf', 'subject': '', 'total_pages': 4114, 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'modDate': 'D:20260225003940Z', 'keywords': '', 'trapped': '', 'author': '', 'moddate': '2026-02-25T00:39:40+00:00', 'source': '/content/drive/MyDrive/Project5/medical_diagnosis_manual.pdf', 'creationdate': '2012-06-15T05:44:40+00:00', 'creator': 'Atop CHM to PDF Converter', 'creationDate': 'D:20120615054440Z', 'page': 2079}, page_content="Chapter 194. Pulmonary Embolism\nIntroduction\nPulmonary embolism (PE) is the occlusion of ≥ 1 pulmonary arteries by thrombi that originate\nelsewhere, typically in the large veins of the lower extremities or pelvis. Risk factors are\nconditions that impair venous return, conditions that cause endothelial injury or dysfunction,\nand underlying hypercoagulable states. Symptoms

### Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [ ]:
rel_docs = retriever.get_relevant_documents("/content/drive/MyDrive/Project5/medical_diagnosis_manual.pdf")
rel_docs

/tmp/ipykernel_10255/774496280.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever.get_relevant_documents("/content/drive/MyDrive/Project5/medical_diagnosis_manual.pdf")


[Document(metadata={'moddate': '2026-02-25T00:39:40+00:00', 'subject': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'keywords': '', 'total_pages': 4114, 'modDate': 'D:20260225003940Z', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'creator': 'Atop CHM to PDF Converter', 'trapped': '', 'file_path': '/content/drive/MyDrive/Project5/medical_diagnosis_manual.pdf', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'author': '', 'format': 'PDF 1.7', 'page': 3436, 'source': '/content/drive/MyDrive/Project5/medical_diagnosis_manual.pdf', 'creationDate': 'D:20120615054440Z'}, page_content='The Merck Manual of Diagnosis & Therapy, 19th Edition\nChapter 329. Burns\n3427\nchadnar@aim.com\nYUF3JRZ527\nThis file is meant for personal use by chadnar@aim.com only.\nSharing or publishing the contents in part or full is liable for legal action.'),
 Document(metadata={'subject': '', 'format': 'PDF 1.7', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 't

In [ ]:
#optimize fix here
model_output = llm(
      "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
      max_tokens=5,
      temperature=0.3
    )

Llama.generate: prefix-match hit


In [ ]:
model_output['choices'][0]['text']

'\n\nSudden'

### System and User Prompt Template

chunked dos (1024) -> sentence transformer -> Chroma DB (contains vectorized information of these chunks).

Have a grounded prompt (user query + retrieved

In [ ]:
qna_system_message = """
You are an AI assistant whose work is to review the report and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Include the source reference with page numbers, journal name, or publication as provided in the context.
Do not provide additional information outside the context.

If the answer is not found in the context, respond "I don't know".
Do not provide information outside of the context,

Here is a an example of how to structure your repsonse:
Answer:
[Medical answer based on context]

Source:
[Source details with page or section]
"""

qna_user_message_template = """
###Context
Here are some documents that are relevant to the medical question mentioned below.
{context}

###Question
{question}
"""

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

RAG
user_message:= (query + retrieved context)
grounded_prompt= user_message+ system_message

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_1,top_k=20)

Llama.generate: prefix-match hit


'Answer:\nThe protocol for managing sepsis in a critical care unit involves obtaining cultures of blood and any other appropriate specimens, administering empiric antibiotics after cultures are obtained, adjusting antibiotics according to culture and susceptibility testing results, surgically draining any abscesses, removing internal devices that may be the source of bacteria, and providing supportive care such as fluids, antipyretics, analgesics, and oxygen for patients with hypoxemia. (Source: Chapter 222, pages 3480-3481)'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symtoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2, top_k=20)

Llama.generate: prefix-match hit


"Answer:\nThe common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs include pain felt in the right lower quadrant with palpation of the left lower quadrant"

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3, top_k=20)

Llama.generate: prefix-match hit


'Answer:\nThe treatment options for sudden patchy hair loss, also known as alopecia areata, include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA) (Table 86-2). The cause of alopecia areata is believed to be an autoimmune disorder affecting genetically susceptible individuals exposed to unclear environmental'

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4, top_k=20)

Llama.generate: prefix-match hit


'Answer:\nInitial treatment for a person with a traumatic brain injury (TBI) includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation.\n\nSource'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5, top_k=20)

Llama.generate: prefix-match hit


'Answer:\nFor a person who has fractured their leg during a hiking trip, the first step is to immobilize the injury immediately by splinting to prevent further damage. Pain should be treated with opioids. Definitive treatment often involves reduction, which may require analgesia or sedation. Closed reduction is maintained by casting, while open reduction is maintained by surgical hardware such as pins, screws, plates, or external fixators. RICE (rest, ice, compression, and elevation) should be applied to the injury for the first 24 to 48 hours'

### Fine-tuning

In [ ]:
#1. What is the protocol for managing sepsis in a critical care unit?
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_1,temperature=0.5)

Llama.generate: prefix-match hit


'Answer:\nSuspected cases of bacteremia, sepsis, or septic shock in a critical care unit require cultures and empiric antibiotics as early treatment (Merck Manual of Diagnosis & Therapy, 19th Edition, p. 2391). Patients should have their antibiotics adjusted based on culture and susceptibility testing, surgical intervention for abscesses, and removal of internal devices if necessary.\n\nSource:\nThe Merck Manual of Diagnosis & Therapy, 19th Edition, p. 23'

In [ ]:
#2. What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,temperature=0.5, top_p=0.8)

Llama.generate: prefix-match hit


"Answer:\nThe common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs include pain felt in the right lower quadrant with palpation of the left lower quadrant"

In [ ]:
#3. What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?
user_input_3="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3,temperature=0.5, k=4)

Llama.generate: prefix-match hit


'Answer:\nThe treatment for sudden patchy hair loss, also known as alopecia areata, includes topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA) (Merck Manual of Diagnosis & Therapy, 19th Edition, p. 849). The possible causes behind this condition are autoimmune disorders and un'

In [ ]:
#4. What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
user_input_4="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4,temperature=0.5, top_p=0.8, top_k=20)

Llama.generate: prefix-match hit


'Answer:\nInitial treatment for a person with a traumatic brain injury (TBI) includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation (The Merck'

In [ ]:
#5.  What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?
user_input_5=" What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5,temperature=0.5, top_p=0.5, top_k=30)

Llama.generate: prefix-match hit


'Answer:\nFor a person who has fractured their leg during a hiking trip, the first step is to immobilize the injury immediately by splinting. This helps prevent further injury and decreases pain. Definitive treatment often involves reduction, which may require analgesia or sedation. Closed reduction is maintained by casting, while open reduction is maintained by various surgical hardware. RICE (rest, ice, compression, elevation) is beneficial for soft-tissue injuries. Pain is typically treated with opioids. Hemorrhagic shock from arterial injuries is treated surgically unless'

## Output Evaluation - start here

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answers generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message  = """You are tasked with rating medical AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score."""

In [ ]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""
#Evaluate based on grounness and relevance
    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
qna_system_message = """
You are an AI assistant whose work is to review the report and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Include the source reference with page numbers, journal name, or publication as provided in the context.
Do not provide additional information outside the context.

If the answer is not found in the context, respond "I don't know".
Do not provide information outside of the context,

Here is a an example of how to structure your repsonse:
Answer:
[Medical answer based on context]

Source:
[Source details with page or section]
"""

qna_user_message_template = """
###Context
Here are some documents that are relevant to the medical question mentioned below.
{context}

###Question
{question}
"""

ground,rel = generate_ground_relevance_response(user_input="What is the protocol for managing sepsis in a critical care unit?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the key information in the context related to managing sepsis in a critical care unit.
2. Determine if the AI generated answer is derived only from the information presented in the context.
3. Compare each element of the answer with the corresponding information in the context.

Evaluation:
The AI generated answer adheres to the metric as it is derived solely from the information provided in the context. The context discusses the importance of early detection and treatment of sepsis, including obtaining cultures and administering empiric antibiotics based on suspected bacteremia. The AI generated answer accurately reflects these steps and does not include any additional or irrelevant information.

Rating:
Based on the evaluation criteria, I would rate this answer as a 5 - The metric is followed completely.

 Steps to evaluate context as per relevance metric:
1. Identify the main aspects of the question: managing sepsis in a critical care unit.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to appendicitis symptoms and treatment.
2. Check if the answer is derived only from the information presented in the context.

Explanation:
The answer adheres to the metric as it is derived solely from the information provided in the context. The context discusses the symptoms of appendicitis, which are detailed in the answer, and also mentions that treatment involves surgical removal of the appendix. Therefore, the answer accurately reflects the information presented in the context.

Evaluation:
The metric is followed completely.

Rating:
Based on the evaluation criteria, I would rate the answer a 5.

 Steps to evaluate the context as per the metric:
1. Identify the main aspects of the question: common symptoms for appendicitis and whether it can be cured via medicine or not.
2. Determine if the context provides relevant information about these aspects.
3. Check if all important aspects are contained in 

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the question and context.
2. Understand the definition of alopecia areata and its causes as given in the context.
3. Determine if the answer mentions only information derived from the context.
4. Check if the treatment options mentioned in the answer match those listed in the context.
5. Verify that the possible causes mentioned in the answer correspond to those stated in the context.

Explanation:
The question asks about effective treatments and possible causes for sudden patchy hair loss, which is a description of alopecia areata as per the context. The answer mentions treatment options such as topical or intralesional corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), and psoralen plus ultraviolet A (PUVA). These are all listed in the context as potential treatments for alopecia areata.

Furthermore, the answer states that possible causes behind sudden patchy hair loss 

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to treatments for a person with a physical injury to brain tissue.
2. Compare the information in the context to the AI generated answer to ensure that the answer is derived only from the context and not any external sources.

Explanation:
The context provides detailed information about the diagnosis, prognosis, and treatment options for a person with a physical injury to brain tissue. The AI generated answer mentions the initial treatments for such a person which include ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure, and possible surgery in severe cases. These treatments are explicitly mentioned in the context, so it can be concluded that the answer is derived only from the information presented in the context.

Evaluation:
The metric is followed completely as the answer is derived solely from the context provided.

Rating:
Based on the evaluation criteria, 

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to precautions and treatment for a fractured leg.
2. Check if the AI generated answer includes only the information from the context.

Explanation:
The AI generated answer includes all the necessary steps and precautions for treating a person who has fractured their leg, as mentioned in the context. The answer is derived directly from the information presented in the context, so the metric is followed completely.

Rating:
Based on the evaluation criteria, I would rate the AI generated answer as a 5 because it follows the metric completely.

 Steps to evaluate the context as per the relevance metric:
1. Identify the main aspects of the question: necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery.
2. Determine if all important aspects are contained in the context: The context discusses various tre

## Actionable Insights and Business Recommendations

- This project has consecutively refined the output of medical queries from basic AI inquriroes to the calculating the ground releveance of the results.
- The basic medical inquiries only yielded basic results which almost anybody could have told you.
- Initially the results were generic, but through the use of RAG, they were much more refined, referencing the Merck manual. Fine tuning did not appear to help too much here, although searchiing for the ground truth did refine the end results.
- With prompt engineering, the results became more valid with the system prompt providing context.
- Data was then loaded with RAG, chunked, embedded, vectorized, and fed into a retriever working with system and user prompt templates. This enhanced the output results.
- Fine tuninig tried to rework the results with the use of the temperature variable, but temperature only added to the hallucinitive effect so did not assist much.
- Finally, I used the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. The evaluation was based on the answers generated to the questions from the previous section. A groundness_rater_system_message and a relevance_rater_system_message were included.
- Various medical ailments could be effectively diagnosed using AI.
- It would appear this tool would work well, alongside a human doctor to actually talk with the patient who desires human contact.

Things to try:
1. Enchance Contextual Relevance : Increase the chunk_over parameter in the retriever to improve result relevance. Sinced the medical manual contains sequential instructions, a higher overlap will result in more context continuity.
2. Maintain High Groundness: The model achieved a full score in groundness due to strict prompting.
3. Optimize Embeddings for Domain Specific Accuracy : Using a model trained on medical datasets should enhance the rsults.
4. Continous Knowledge Update : Regularly update the knowledge base to stay relevant.
5. Feedback Integration : Include a feedback mechanism for doctors to be able to refine the chatbot's responses to respond to real world medical situations.
6. Incorporate other Specializations : Expand the RAG system to support additional medical specialties, broadening its utility across healthcare.
7. Code really requires a paid tier of Google Colab to run properly. It timed out withe the free tier.

<font size=6 color='blue'>Power Ahead</font>
___

In [ ]:
import json
import subprocess

notebook_path = "/content/drive/MyDrive/Project5/Final-Full_Code_NLP_RAG_Project_Notebook_.ipynb"
clean_path = "/content/drive/MyDrive/Project5/clean_Final-Full_Code_NLP_RAG_Project_Notebook_.ipynb"

# Load notebook
with open(notebook_path, 'r') as f:
    nb = json.load(f)

# Remove widget metadata that's causing the error
if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']
    print("Widget metadata removed!")

# Save clean version
with open(clean_path, 'w') as f:
    json.dump(nb, f)
print("Clean notebook saved!")

# Convert clean version to HTML
result = subprocess.run(
    ["jupyter", "nbconvert", "--to", "html", clean_path],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("Output:", result.stdout)
print("Error:", result.stderr)

Widget metadata removed!
Clean notebook saved!
Return code: 0
Output: 
Error: [NbConvertApp] Converting notebook /content/drive/MyDrive/Project5/clean_Final-Full_Code_NLP_RAG_Project_Notebook_.ipynb to html
[NbConvertApp] Writing 1488305 bytes to /content/drive/MyDrive/Project5/clean_Final-Full_Code_NLP_RAG_Project_Notebook_.html



In [ ]:
import shutil

# Copy HTML to Google Drive with correct filename
src  = "/content/drive/MyDrive/Project5/clean_Final-Full_Code_NLP_RAG_Project_Notebook_.html"
dst  = "/content/drive/MyDrive/Project5/Final-Full_Code_NLP_RAG_Project_Notebook_.html"

shutil.copy(src, dst)
print("HTML saved to Google Drive!")
print(f"Location: {dst}")

HTML saved to Google Drive!
Location: /content/drive/MyDrive/Project5/Final-Full_Code_NLP_RAG_Project_Notebook_.html


In [ ]:
from google.colab import runtime
runtime.unassign()
